In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/ko/test.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/ko/train.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/ko/dev.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sw/test.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sw/train.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sw/dev.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sv/test.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sv/train.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/sv/dev.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/da/test.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/da/train.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/da/dev.txt
/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/zh/clean_al

In [2]:
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# 1. Load Data từ clear_all.txt
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            parts = line.strip().split("####")
            if len(parts) < 2: continue
            sent, labels = parts[0], eval(parts[1])
            triples = [f"{a}:{c}:{s}" for a, c, s in labels]
            inputs.append("uabsa: " + sent)
            targets.append(" ; ".join(triples))
    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/en/clean_all.txt")

# 2. Split dataset (80% train, 10% valid, 10% test)
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": train_test["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"]
})

# 3. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = dataset["train"].map(tokenize, batched=True)
valid_dataset = dataset["validation"].map(tokenize, batched=True)
test_dataset = dataset["test"]

# 4. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

# 5. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 6. Training args
training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 8. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_dataset:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# 9. In thử
for i in range(5):
    print(f"Input: {test_dataset[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Epoch,Training Loss,Validation Loss
1,No log,5.005729
2,17.987911,1.618093
3,3.349730,1.212363
4,1.832107,1.020239
5,1.365456,0.890152
6,1.161255,0.884621
7,0.974511,0.865041
8,0.937214,0.887325
9,0.857783,0.870533
10,0.836366,0.857643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: uabsa: as somebody who studies management, i have grasped an unreal amout of knowledge regarding financial markets.
Pred : NULL:course general:positive

Input: uabsa: that being said, it does seem that they could have done a much better job in the video introducing core concepts that would have given non - programmers a better starting point for looking for outside researchers.
Pred : video:presentation quality:negative

Input: uabsa: test are needed.
Pred : test:assignments quality:negative

Input: uabsa: the course material is framed in a comprehensive manner.
Pred : material:material quality:positive

Input: uabsa: thanks this course i found where i am doing wrong and actually i´am learning new things easier and time after time i repeat the information that i have studied some years ago.
Pred : course:course general:positive ; course:course general:positive



In [3]:
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_dataset]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Baseline
Precision: 0.5301
Recall:    0.4554
F1-Score:  0.4899


In [4]:
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# 1. Load Data từ clear_all.txt
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            parts = line.strip().split("####")
            if len(parts) < 2: continue
            sent, labels = parts[0], eval(parts[1])
            triples = [f"{a}:{c}:{s}" for a, c, s in labels]
            inputs.append("uabsa: " + sent)
            targets.append(" ; ".join(triples))
    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/vi/clean_all.txt")

# 2. Split dataset (80% train, 10% valid, 10% test)
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": train_test["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"]
})

# 3. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = dataset["train"].map(tokenize, batched=True)
valid_dataset = dataset["validation"].map(tokenize, batched=True)
test_dataset = dataset["test"]

# 4. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

# 5. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 6. Training args
training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 8. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_dataset:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# 9. In thử
for i in range(5):
    print(f"Input: {test_dataset[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

Map:   0%|          | 0/1410 [00:00<?, ? examples/s]

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Epoch,Training Loss,Validation Loss
1,22.226963,3.779409
2,3.342684,1.947705
3,2.072399,1.684324
4,1.690297,1.691977
5,1.547780,1.576071
6,1.432194,1.514435
7,1.344869,1.522291
8,1.314770,1.446373
9,1.282947,1.350100
10,1.197557,1.272094


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: uabsa: tôi muốn giới thiệu khóa học này cho những nhà phát triển hiện tại không có kiến thức nền về khoa học máy tính.
Pred : khóa học:course general:positive

Input: uabsa: khóa học này thật tuyệt vời.
Pred : khóa học:course general:positive

Input: uabsa: tôi rất thích khóa học này khóa học.
Pred : khóa học:course general:positive

Input: uabsa: đảm bảo dành đủ thời gian cho việc đọc và làm bài tập chính.
Pred : NULL:course general:positive

Input: uabsa: tôi thực sự thích khóa học. thông thường, học trực tuyến khóa học khá khó khăn với tôi. nhưng không phải với khóa học này.
Pred : khóa học:course general:positive



In [5]:
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_dataset]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Baseline
Precision: 0.3559
Recall:    0.2788
F1-Score:  0.3127


In [6]:
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# 1. Load Data từ clear_all.txt
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line in f:
            parts = line.strip().split("####")
            if len(parts) < 2: continue
            sent, labels = parts[0], eval(parts[1])
            triples = [f"{a}:{c}:{s}" for a, c, s in labels]
            inputs.append("uabsa: " + sent)
            targets.append(" ; ".join(triples))
    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/zh/clean_all.txt")

# 2. Split dataset (80% train, 10% valid, 10% test)
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

dataset = DatasetDict({
    "train": train_test["train"],
    "validation": valid_test["train"],
    "test": valid_test["test"]
})

# 3. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = dataset["train"].map(tokenize, batched=True)
valid_dataset = dataset["validation"].map(tokenize, batched=True)
test_dataset = dataset["test"]

# 4. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

# 5. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 6. Training args
training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 8. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_dataset:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# 9. In thử
for i in range(5):
    print(f"Input: {test_dataset[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

Map:   0%|          | 0/1411 [00:00<?, ? examples/s]

Parameter 'function'=<function tokenize at 0x7debe3226b60> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: uabsa: 我会向没有计算机科学背景的现有开发人员推荐这个课程。
Pred : 课程:course general:positive

Input: uabsa: 这课程很棒。
Pred : 课程:course general:positive

Input: uabsa: 我非常喜欢这个课程。
Pred : 课程:course general:positive

Input: uabsa: 我真的很喜欢这门课。
Pred : 课:course general:positive

Input: uabsa: 就记住商业和创业概念而言，这课程对我非常有用。
Pred : 课程:course general:positive



In [7]:
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_dataset]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Baseline
Precision: 0.5297
Recall:    0.4375
F1-Score:  0.4792


In [8]:
import torch
import ast
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForSeq2Seq
)

# 1. Load Data
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line_num, line in enumerate(f):
            parts = line.strip().split("####")
            if len(parts) < 2:
                continue

            sent = parts[0]
            label_str = parts[1].strip()

            try:
                # Dùng ast.literal_eval an toàn hơn eval
                labels = ast.literal_eval(label_str)

                # Tạo triples: aspect:category:sentiment
                triples = [f"{a}:{c}:{s}" for a, c, s in labels]

                inputs.append("trích xuất khía cạnh cảm xúc bộ ba: " + sent)
                targets.append(" ; ".join(triples))
            except Exception:
                print(f"Lỗi tại dòng {line_num}: {label_str}")
                continue

    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

# Đọc toàn bộ dữ liệu từ clean_all.txt
all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/vi/clean_all.txt")

# Tự chia train / valid / test: 80% / 10% / 10%
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

train = train_test["train"]
valid = valid_test["train"]
test = valid_test["test"]

# 2. Tokenizer & Map
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train.map(tokenize, batched=True)
valid_dataset = valid.map(tokenize, batched=True)

# 3. Model & Trainer
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")

# Tắt cảnh báo tied weights
model.config.tie_word_embeddings = False

# Data collator sẽ tự động xử lý padding trong labels thành -100
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 4. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# In thử 5 câu đầu
for i in range(5):
    print(f"Input: {test[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

# 5. Tính Precision / Recall / F1
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Map:   0%|          | 0/1410 [00:00<?, ? examples/s]

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Epoch,Training Loss,Validation Loss
1,19.537872,1.764713
2,2.232499,1.143164
3,1.414415,0.989095
4,1.116599,0.920231
5,0.991070,0.800631
6,0.859109,0.746789
7,0.789455,0.776014
8,0.733609,0.713642
9,0.701745,0.704649
10,0.673764,0.700343


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: trích xuất khía cạnh cảm xúc bộ ba: tôi muốn giới thiệu khóa học này cho những nhà phát triển hiện tại không có kiến thức nền về khoa học máy tính.
Pred : khóa học:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: khóa học này thật tuyệt vời.
Pred : khóa học:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: tôi rất thích khóa học này khóa học.
Pred : khóa học:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: đảm bảo dành đủ thời gian cho việc đọc và làm bài tập chính.
Pred : bài tập chính:assignments quality:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: tôi thực sự thích khóa học. thông thường, học trực tuyến khóa học khá khó khăn với tôi. nhưng không phải với khóa học này.
Pred : khóa học:course general:positive

Baseline
Precision: 0.4574
Recall:    0.3805
F1-Score:  0.4155


In [9]:
import torch
import ast
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForSeq2Seq
)

# 1. Load Data
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line_num, line in enumerate(f):
            parts = line.strip().split("####")
            if len(parts) < 2:
                continue

            sent = parts[0]
            label_str = parts[1].strip()

            try:
                # Dùng ast.literal_eval an toàn hơn eval
                labels = ast.literal_eval(label_str)

                # Tạo triples: aspect:category:sentiment
                triples = [f"{a}:{c}:{s}" for a, c, s in labels]

                inputs.append("trích xuất khía cạnh cảm xúc bộ ba: " + sent)
                targets.append(" ; ".join(triples))
            except Exception:
                print(f"Lỗi tại dòng {line_num}: {label_str}")
                continue

    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

# Đọc toàn bộ dữ liệu từ clean_all.txt
all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/en/clean_all.txt")

# Tự chia train / valid / test: 80% / 10% / 10%
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

train = train_test["train"]
valid = valid_test["train"]
test = valid_test["test"]

# 2. Tokenizer & Map
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train.map(tokenize, batched=True)
valid_dataset = valid.map(tokenize, batched=True)

# 3. Model & Trainer
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")

# Tắt cảnh báo tied weights
model.config.tie_word_embeddings = False

# Data collator sẽ tự động xử lý padding trong labels thành -100
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 4. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# In thử 5 câu đầu
for i in range(5):
    print(f"Input: {test[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

# 5. Tính Precision / Recall / F1
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Map:   0%|          | 0/1414 [00:00<?, ? examples/s]

Map:   0%|          | 0/177 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Epoch,Training Loss,Validation Loss
1,22.268813,2.791191
2,2.918502,1.508788
3,1.747245,1.130965
4,1.241906,0.939541
5,1.045286,0.936020
6,0.946829,0.864370
7,0.815825,0.878758
8,0.774853,0.876415
9,0.716792,0.862804
10,0.686801,0.847532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: trích xuất khía cạnh cảm xúc bộ ba: as somebody who studies management, i have grasped an unreal amout of knowledge regarding financial markets.
Pred : NULL:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: that being said, it does seem that they could have done a much better job in the video introducing core concepts that would have given non - programmers a better starting point for looking for outside researchers.
Pred : video:presentation quality:negative

Input: trích xuất khía cạnh cảm xúc bộ ba: test are needed.
Pred : Test:assignments quality:negative

Input: trích xuất khía cạnh cảm xúc bộ ba: the course material is framed in a comprehensive manner.
Pred : material:material quality:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: thanks this course i found where i am doing wrong and actually i´am learning new things easier and time after time i repeat the information that i have studied some years ago.
Pred : course:course general:positive ; course

In [10]:
import torch
import ast
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    MT5ForConditionalGeneration, 
    Trainer, 
    TrainingArguments, 
    DataCollatorForSeq2Seq
)

# 1. Load Data
def load_absa(path):
    inputs, targets = [], []
    with open(path, "r", encoding="utf8") as f:
        for line_num, line in enumerate(f):
            parts = line.strip().split("####")
            if len(parts) < 2:
                continue

            sent = parts[0]
            label_str = parts[1].strip()

            try:
                # Dùng ast.literal_eval an toàn hơn eval
                labels = ast.literal_eval(label_str)

                # Tạo triples: aspect:category:sentiment
                triples = [f"{a}:{c}:{s}" for a, c, s in labels]

                inputs.append("trích xuất khía cạnh cảm xúc bộ ba: " + sent)
                targets.append(" ; ".join(triples))
            except Exception:
                print(f"Lỗi tại dòng {line_num}: {label_str}")
                continue

    return Dataset.from_dict({"input_text": inputs, "target_text": targets})

# Đọc toàn bộ dữ liệu từ clean_all.txt
all_data = load_absa("/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/zh/clean_all.txt")

# Tự chia train / valid / test: 80% / 10% / 10%
train_test = all_data.train_test_split(test_size=0.2, seed=42)
valid_test = train_test["test"].train_test_split(test_size=0.5, seed=42)

train = train_test["train"]
valid = valid_test["train"]
test = valid_test["test"]

# 2. Tokenizer & Map
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(example["input_text"], max_length=128, truncation=True)
    labels = tokenizer(text_target=example["target_text"], max_length=64, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train.map(tokenize, batched=True)
valid_dataset = valid.map(tokenize, batched=True)

# 3. Model & Trainer
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")

# Tắt cảnh báo tied weights
model.config.tie_word_embeddings = False

# Data collator sẽ tự động xử lý padding trong labels thành -100
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=5e-4,
    num_train_epochs=10,
    logging_steps=50,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()

# 4. Inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test:
    inputs = tokenizer(sample["input_text"], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    preds.append(pred)

# In thử 5 câu đầu
for i in range(5):
    print(f"Input: {test[i]['input_text']}")
    print(f"Pred : {preds[i]}\n")

# 5. Tính Precision / Recall / F1
def extract_triples(text):
    triples = set()
    parts = text.split(";")
    for p in parts:
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct = 0
    n_pred = 0
    n_gold = 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)

        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test]

metrics = calculate_f1(preds, gold_targets)

print("Baseline")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")

Map:   0%|          | 0/1411 [00:00<?, ? examples/s]

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warni

Epoch,Training Loss,Validation Loss
1,25.231345,7.394665
2,4.144355,1.850416
3,1.792157,1.430983
4,1.419564,1.111575
5,1.215050,0.985120
6,0.982832,0.911394
7,0.913207,0.866455
8,0.782821,0.836511
9,0.788995,0.807368
10,0.727504,0.793277


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Input: trích xuất khía cạnh cảm xúc bộ ba: 我会向没有计算机科学背景的现有开发人员推荐这个课程。
Pred : 课程:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: 这课程很棒。
Pred : 课程:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: 我非常喜欢这个课程。
Pred : 课程:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: 我真的很喜欢这门课。
Pred : 课:course general:positive

Input: trích xuất khía cạnh cảm xúc bộ ba: 就记住商业和创业概念而言，这课程对我非常有用。
Pred : 课程:course general:positive

Baseline
Precision: 0.4947
Recall:    0.4196
F1-Score:  0.4541


In [11]:
import os
import ast
import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Đọc tất cả file clean_all.txt theo từng domain
def load_all_clean_data(base_path):
    all_data = []

    domains = ["laptop", "hotel", "coursera", "sight", "phone", "restaurant", "food"]

    def parse_file(path):
        inputs, targets = [], []

        if not os.path.exists(path):
            print(f"Không tìm thấy file: {path}")
            return Dataset.from_dict({"input_text": [], "target_text": []})

        with open(path, "r", encoding="utf8") as f:
            for line_num, line in enumerate(f, 1):
                parts = line.strip().split("####")
                if len(parts) < 2:
                    continue

                sent = parts[0].strip()
                label_str = parts[1].strip()

                try:
                    labels = ast.literal_eval(label_str)
                    triples = [f"{a}:{c}:{s}" for a, c, s in labels]
                    inputs.append("extract aspect sentiment triples: " + sent)
                    targets.append(" ; ".join(triples))
                except Exception as e:
                    print(f"Lỗi parse tại {path}, dòng {line_num}: {e}")
                    continue

        return Dataset.from_dict({
            "input_text": inputs,
            "target_text": targets
        })

    for domain in domains:
        file_path = f"{base_path}/{domain}/en/clean_all.txt"
        print(f"Loading domain: {domain} -> {file_path}")
        dataset = parse_file(file_path)
        print(f"Số mẫu của {domain}: {len(dataset)}")
        all_data.append(dataset)

    merged_dataset = concatenate_datasets(all_data)
    return merged_dataset


# 2. Chia dữ liệu train / valid / test
def split_dataset(dataset, test_size=0.1, valid_size=0.1, seed=42):
    # Bước 1: tách test trước
    split_1 = dataset.train_test_split(test_size=test_size, seed=seed)
    train_valid = split_1["train"]
    test_data = split_1["test"]

    # Bước 2: tách valid từ phần train_valid
    # valid_size đang tính theo toàn bộ dataset, nên cần quy đổi lại
    valid_ratio_adjusted = valid_size / (1 - test_size)

    split_2 = train_valid.train_test_split(test_size=valid_ratio_adjusted, seed=seed)
    train_data = split_2["train"]
    valid_data = split_2["test"]

    return train_data, valid_data, test_data


# 3. Load toàn bộ clean_all.txt
base_data_path = "/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data"
full_dataset = load_all_clean_data(base_data_path)

print(f"\nTổng số mẫu sau khi gộp: {len(full_dataset)}")

# 4. Chia dữ liệu
train_data, valid_data, test_data = split_dataset(
    full_dataset,
    test_size=0.1,   # 10% test
    valid_size=0.1,  # 10% validation
    seed=42
)

print(f"Train: {len(train_data)}")
print(f"Valid: {len(valid_data)}")
print(f"Test : {len(test_data)}")


# 5. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True
    )
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_data.map(tokenize, batched=True)
valid_dataset = valid_data.map(tokenize, batched=True)


# 6. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5_clean_all",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    logging_dir="./logs",
    logging_steps=50
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()


# 7. Inference trên test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_data:
    inputs = tokenizer(sample["input_text"], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))


# 8. Tính metric
def extract_triples(text):
    triples = set()
    for p in text.split(";"):
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct, n_pred, n_gold = 0, 0, 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)
        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_data]
results = calculate_f1(preds, gold_targets)

print("\n--- KẾT QUẢ TRÊN TẬP CLEAN_ALL ĐÃ CHIA LẠI ---")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")

Loading domain: laptop -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/en/clean_all.txt
Số mẫu của laptop: 2122
Loading domain: hotel -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/hotel/en/clean_all.txt
Số mẫu của hotel: 1729
Loading domain: coursera -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/en/clean_all.txt
Số mẫu của coursera: 1768
Loading domain: sight -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/sight/en/clean_all.txt
Số mẫu của sight: 1569
Loading domain: phone -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/phone/en/clean_all.txt
Số mẫu của phone: 1778
Loading domain: restaurant -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/restaurant/en/clean_all.txt
Số mẫu của restaurant: 2124
Loading domain: food -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/food/en/clean_all.txt
Số mẫu của food: 1647

Tổng số mẫu sau khi gộp: 12737
Train: 10189
Valid: 1274
Test : 1274


Map:   0%|          | 0/10189 [00:00<?, ? examples/s]

Map:   0%|          | 0/1274 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather alo

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- KẾT QUẢ TRÊN TẬP CLEAN_ALL ĐÃ CHIA LẠI ---
Precision: 0.3806
Recall:    0.3389
F1-Score:  0.3585


In [12]:
import os
import ast
import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Đọc tất cả file clean_all.txt theo từng domain
def load_all_clean_data(base_path):
    all_data = []

    domains = ["laptop", "hotel", "coursera", "sight", "phone", "restaurant", "food"]

    def parse_file(path):
        inputs, targets = [], []

        if not os.path.exists(path):
            print(f"Không tìm thấy file: {path}")
            return Dataset.from_dict({"input_text": [], "target_text": []})

        with open(path, "r", encoding="utf8") as f:
            for line_num, line in enumerate(f, 1):
                parts = line.strip().split("####")
                if len(parts) < 2:
                    continue

                sent = parts[0].strip()
                label_str = parts[1].strip()

                try:
                    labels = ast.literal_eval(label_str)
                    triples = [f"{a}:{c}:{s}" for a, c, s in labels]
                    inputs.append("extract aspect sentiment triples: " + sent)
                    targets.append(" ; ".join(triples))
                except Exception as e:
                    print(f"Lỗi parse tại {path}, dòng {line_num}: {e}")
                    continue

        return Dataset.from_dict({
            "input_text": inputs,
            "target_text": targets
        })

    for domain in domains:
        file_path = f"{base_path}/{domain}/vi/clean_all.txt"
        print(f"Loading domain: {domain} -> {file_path}")
        dataset = parse_file(file_path)
        print(f"Số mẫu của {domain}: {len(dataset)}")
        all_data.append(dataset)

    merged_dataset = concatenate_datasets(all_data)
    return merged_dataset


# 2. Chia dữ liệu train / valid / test
def split_dataset(dataset, test_size=0.1, valid_size=0.1, seed=42):
    # Bước 1: tách test trước
    split_1 = dataset.train_test_split(test_size=test_size, seed=seed)
    train_valid = split_1["train"]
    test_data = split_1["test"]

    # Bước 2: tách valid từ phần train_valid
    # valid_size đang tính theo toàn bộ dataset, nên cần quy đổi lại
    valid_ratio_adjusted = valid_size / (1 - test_size)

    split_2 = train_valid.train_test_split(test_size=valid_ratio_adjusted, seed=seed)
    train_data = split_2["train"]
    valid_data = split_2["test"]

    return train_data, valid_data, test_data


# 3. Load toàn bộ clean_all.txt
base_data_path = "/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data"
full_dataset = load_all_clean_data(base_data_path)

print(f"\nTổng số mẫu sau khi gộp: {len(full_dataset)}")

# 4. Chia dữ liệu
train_data, valid_data, test_data = split_dataset(
    full_dataset,
    test_size=0.1,   # 10% test
    valid_size=0.1,  # 10% validation
    seed=42
)

print(f"Train: {len(train_data)}")
print(f"Valid: {len(valid_data)}")
print(f"Test : {len(test_data)}")


# 5. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True
    )
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_data.map(tokenize, batched=True)
valid_dataset = valid_data.map(tokenize, batched=True)


# 6. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5_clean_all",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    logging_dir="./logs",
    logging_steps=50
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()


# 7. Inference trên test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_data:
    inputs = tokenizer(sample["input_text"], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))


# 8. Tính metric
def extract_triples(text):
    triples = set()
    for p in text.split(";"):
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct, n_pred, n_gold = 0, 0, 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)
        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_data]
results = calculate_f1(preds, gold_targets)

print("\n--- KẾT QUẢ TRÊN TẬP CLEAN_ALL (VI)---")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")

Loading domain: laptop -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/vi/clean_all.txt
Số mẫu của laptop: 2117
Loading domain: hotel -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/hotel/vi/clean_all.txt
Số mẫu của hotel: 1723
Loading domain: coursera -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/vi/clean_all.txt
Số mẫu của coursera: 1763
Loading domain: sight -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/sight/vi/clean_all.txt
Số mẫu của sight: 1569
Loading domain: phone -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/phone/vi/clean_all.txt
Số mẫu của phone: 1774
Loading domain: restaurant -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/restaurant/vi/clean_all.txt
Số mẫu của restaurant: 2115
Loading domain: food -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/food/vi/clean_all.txt
Số mẫu của food: 1636

Tổng số mẫu sau khi gộp: 12697
Train: 10157
Valid: 1270
Test : 1270


Map:   0%|          | 0/10157 [00:00<?, ? examples/s]

Map:   0%|          | 0/1270 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather alo

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- KẾT QUẢ TRÊN TẬP CLEAN_ALL (VI)---
Precision: 0.3242
Recall:    0.2670
F1-Score:  0.2928


In [13]:
import os
import ast
import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Đọc tất cả file clean_all.txt theo từng domain
def load_all_clean_data(base_path):
    all_data = []

    domains = ["laptop", "hotel", "coursera", "sight", "phone", "restaurant", "food"]

    def parse_file(path):
        inputs, targets = [], []

        if not os.path.exists(path):
            print(f"Không tìm thấy file: {path}")
            return Dataset.from_dict({"input_text": [], "target_text": []})

        with open(path, "r", encoding="utf8") as f:
            for line_num, line in enumerate(f, 1):
                parts = line.strip().split("####")
                if len(parts) < 2:
                    continue

                sent = parts[0].strip()
                label_str = parts[1].strip()

                try:
                    labels = ast.literal_eval(label_str)
                    triples = [f"{a}:{c}:{s}" for a, c, s in labels]
                    inputs.append("extract aspect sentiment triples: " + sent)
                    targets.append(" ; ".join(triples))
                except Exception as e:
                    print(f"Lỗi parse tại {path}, dòng {line_num}: {e}")
                    continue

        return Dataset.from_dict({
            "input_text": inputs,
            "target_text": targets
        })

    for domain in domains:
        file_path = f"{base_path}/{domain}/zh/clean_all.txt"
        print(f"Loading domain: {domain} -> {file_path}")
        dataset = parse_file(file_path)
        print(f"Số mẫu của {domain}: {len(dataset)}")
        all_data.append(dataset)

    merged_dataset = concatenate_datasets(all_data)
    return merged_dataset


# 2. Chia dữ liệu train / valid / test
def split_dataset(dataset, test_size=0.1, valid_size=0.1, seed=42):
    # Bước 1: tách test trước
    split_1 = dataset.train_test_split(test_size=test_size, seed=seed)
    train_valid = split_1["train"]
    test_data = split_1["test"]

    # Bước 2: tách valid từ phần train_valid
    # valid_size đang tính theo toàn bộ dataset, nên cần quy đổi lại
    valid_ratio_adjusted = valid_size / (1 - test_size)

    split_2 = train_valid.train_test_split(test_size=valid_ratio_adjusted, seed=seed)
    train_data = split_2["train"]
    valid_data = split_2["test"]

    return train_data, valid_data, test_data


# 3. Load toàn bộ clean_all.txt
base_data_path = "/kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data"
full_dataset = load_all_clean_data(base_data_path)

print(f"\nTổng số mẫu sau khi gộp: {len(full_dataset)}")

# 4. Chia dữ liệu
train_data, valid_data, test_data = split_dataset(
    full_dataset,
    test_size=0.1,   # 10% test
    valid_size=0.1,  # 10% validation
    seed=42
)

print(f"Train: {len(train_data)}")
print(f"Valid: {len(valid_data)}")
print(f"Test : {len(test_data)}")


# 5. Tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

def tokenize(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=128,
        truncation=True
    )
    labels = tokenizer(
        text_target=example["target_text"],
        max_length=64,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_data.map(tokenize, batched=True)
valid_dataset = valid_data.map(tokenize, batched=True)


# 6. Model
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-small")
model.config.tie_word_embeddings = False

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./uabsa_mt5_clean_all",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    push_to_hub=False,
    logging_dir="./logs",
    logging_steps=50
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
)

trainer.train()


# 7. Inference trên test
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

preds = []
for sample in test_data:
    inputs = tokenizer(sample["input_text"], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=64)
    preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))


# 8. Tính metric
def extract_triples(text):
    triples = set()
    for p in text.split(";"):
        p = p.strip()
        if p:
            triples.add(p)
    return triples

def calculate_f1(preds, targets):
    n_correct, n_pred, n_gold = 0, 0, 0

    for p, g in zip(preds, targets):
        p_set = extract_triples(p)
        g_set = extract_triples(g)
        n_correct += len(p_set & g_set)
        n_pred += len(p_set)
        n_gold += len(g_set)

    precision = n_correct / n_pred if n_pred > 0 else 0
    recall = n_correct / n_gold if n_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

gold_targets = [sample["target_text"] for sample in test_data]
results = calculate_f1(preds, gold_targets)

print("\n--- KẾT QUẢ TRÊN TẬP CLEAN_ALL (VI)---")
print(f"Precision: {results['precision']:.4f}")
print(f"Recall:    {results['recall']:.4f}")
print(f"F1-Score:  {results['f1']:.4f}")

Loading domain: laptop -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/laptop/zh/clean_all.txt
Số mẫu của laptop: 2117
Loading domain: hotel -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/hotel/zh/clean_all.txt
Số mẫu của hotel: 1726
Loading domain: coursera -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/coursera/zh/clean_all.txt
Số mẫu của coursera: 1764
Loading domain: sight -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/sight/zh/clean_all.txt
Số mẫu của sight: 1569
Loading domain: phone -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/phone/zh/clean_all.txt
Số mẫu của phone: 1775
Loading domain: restaurant -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/restaurant/zh/clean_all.txt
Số mẫu của restaurant: 2109
Loading domain: food -> /kaggle/input/datasets/tuongmacvan/m-absb-for-ppnckh/data/food/zh/clean_all.txt
Số mẫu của food: 1641

Tổng số mẫu sau khi gộp: 12701
Train: 10160
Valid: 1270
Test : 1271


Map:   0%|          | 0/10160 [00:00<?, ? examples/s]

Map:   0%|          | 0/1270 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather alo

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- KẾT QUẢ TRÊN TẬP CLEAN_ALL (VI)---
Precision: 0.4113
Recall:    0.3463
F1-Score:  0.3760
